In [2]:
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!nvidia-smi

In [6]:
!unzip -q data.zip -d /content/raw
!ls /content/raw

Ds1  Ds2  Ds3


In [7]:
import yaml
from pathlib import Path
RAW = Path("/content/raw")
if not (RAW/"Ds1").exists() and (RAW/"raw"/"Ds1").exists():
    RAW = RAW/"raw"
for ds in ["Ds1","Ds2","Ds3"]:
    names = yaml.safe_load(open(RAW/ds/"data.yaml"))["names"]
    print(ds, names)

Ds1 ['batteries', 'can', 'electronics', 'others', 'paper', 'pet_bottle', 'plastic_other', 'wet_waste']
Ds2 ['Cardboard', 'Glass', 'Metal', 'Paper', 'Plastic']
Ds3 ['Aerosols', 'Aluminum can', 'Aluminum caps', 'Cardboard', 'Cellulose', 'Ceramic', 'Combined plastic', 'Container for household chemicals', 'Disposable tableware', 'Electronics', 'Foil', 'Furniture', 'Glass bottle', 'Iron utensils', 'Liquid', 'Metal shavings', 'Milk bottle', 'Organic', 'Paper', 'Paper bag', 'Paper cups', 'Paper shavings', 'Papier mache', 'Plastic bag', 'Plastic bottle', 'Plastic can', 'Plastic canister', 'Plastic caps', 'Plastic cup', 'Plastic shaker', 'Plastic shavings', 'Plastic toys', 'Postal packaging', 'Printing industry', 'Scrap metal', 'Stretch film', 'Tetra pack', 'Textile', 'Tin', 'Unknown plastic', 'Wood', 'Zip plastic bag']


In [8]:
import yaml, shutil, re
from pathlib import Path

RAW = Path("/content/raw")
if not (RAW/"Ds1").exists() and (RAW/"raw"/"Ds1").exists():
    RAW = RAW/"raw"
OUT = Path("/content/combined")
if OUT.exists(): shutil.rmtree(OUT)

TARGET = ["paper","cardboard","plastic","plastic_bag","glass","metal","battery","aerosol"]
tid = {n: i for i, n in enumerate(TARGET)}

# first matching rule wins, so the order matters
RULES = [
    (r"aerosol", "aerosol"),
    (r"batter", "battery"),
    (r"paper bag", "paper"),
    (r"plastic bag|\bbag\b", "plastic_bag"),
    (r"glass", "glass"),
    (r"cardboard|carton", "cardboard"),
    (r"paper", "paper"),
    (r"metal|aluminu?m|\bcans?\b|\btin\b|steel", "metal"),
    (r"plastic|bottle|\bpet\b", "plastic"),
]
def target_of(name):
    n = re.sub(r"[_\-]+", " ", name.lower()).strip()
    for pat, t in RULES:
        if re.search(pat, n): return t
    return None

kept = {n: 0 for n in TARGET}
for ds in ["Ds1","Ds2","Ds3"]:
    names = yaml.safe_load(open(RAW/ds/"data.yaml"))["names"]
    if isinstance(names, dict): names = [names[i] for i in sorted(names)]
    old2new = {}
    for i, n in enumerate(names):
        t = target_of(n)
        print(f"{ds}: {n!r} -> {t}")
        if t: old2new[i] = tid[t]
    for split in ["train","valid","test"]:
        imgs = RAW/ds/split/"images"
        if not imgs.exists(): continue
        (OUT/split/"images").mkdir(parents=True, exist_ok=True)
        (OUT/split/"labels").mkdir(parents=True, exist_ok=True)
        for img in imgs.iterdir():
            lbl = RAW/ds/split/"labels"/(img.stem + ".txt")
            lines = []
            if lbl.exists():
                for ln in lbl.read_text().splitlines():
                    p = ln.split()
                    if len(p) == 5 and int(p[0]) in old2new:   # boxes only
                        lines.append(" ".join([str(old2new[int(p[0])])] + p[1:]))
                        kept[TARGET[old2new[int(p[0])]]] += 1
            if not lines: continue
            shutil.copy(img, OUT/split/"images"/f"{ds}_{img.name}")
            (OUT/split/"labels"/f"{ds}_{img.stem}.txt").write_text("\n".join(lines))

yaml.dump({"path": str(OUT), "train": "train/images", "val": "valid/images",
           "test": "test/images", "names": TARGET}, open(OUT/"data.yaml","w"))
print("\nBoxes kept per class:", kept)

Ds1: 'batteries' -> battery
Ds1: 'can' -> metal
Ds1: 'electronics' -> None
Ds1: 'others' -> None
Ds1: 'paper' -> paper
Ds1: 'pet_bottle' -> plastic
Ds1: 'plastic_other' -> plastic
Ds1: 'wet_waste' -> None
Ds2: 'Cardboard' -> cardboard
Ds2: 'Glass' -> glass
Ds2: 'Metal' -> metal
Ds2: 'Paper' -> paper
Ds2: 'Plastic' -> plastic
Ds3: 'Aerosols' -> aerosol
Ds3: 'Aluminum can' -> metal
Ds3: 'Aluminum caps' -> metal
Ds3: 'Cardboard' -> cardboard
Ds3: 'Cellulose' -> None
Ds3: 'Ceramic' -> None
Ds3: 'Combined plastic' -> plastic
Ds3: 'Container for household chemicals' -> None
Ds3: 'Disposable tableware' -> None
Ds3: 'Electronics' -> None
Ds3: 'Foil' -> None
Ds3: 'Furniture' -> None
Ds3: 'Glass bottle' -> glass
Ds3: 'Iron utensils' -> None
Ds3: 'Liquid' -> None
Ds3: 'Metal shavings' -> metal
Ds3: 'Milk bottle' -> plastic
Ds3: 'Organic' -> None
Ds3: 'Paper' -> paper
Ds3: 'Paper bag' -> paper
Ds3: 'Paper cups' -> paper
Ds3: 'Paper shavings' -> paper
Ds3: 'Papier mache' -> None
Ds3: 'Plastic bag' 

In [ ]:
!yolo detect train data=/content/combined/data.yaml model=yolo11n.pt epochs=40 imgsz=640 batch=32 patience=10 project=/content/drive/MyDrive/smart-mrf-runs name=v1

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/combined/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, fr

In [ ]:
!yolo detect predict model=/content/drive/MyDrive/smart-mrf-runs/v1/weights/best.pt source=/content/combined/valid/images save=True

In [ ]:
import glob
from IPython.display import Image, display
for p in glob.glob('/content/runs/detect/predict/*.jpg')[:10]:
    display(Image(filename=p, height=400))

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/smart-mrf-runs/v1/weights/best.pt')